## Load Validated Records

In [0]:
dbutils.widgets.text("ingestion_date", "", "Ingestion Date (YYYY-MM-DD)")
ingestion_date = dbutils.widgets.get("ingestion_date")

if ingestion_date == "":
    from datetime import date
    ingestion_date = str(date.today())

validated_df = spark.table("silver_valuation_multiple_validated") \
    .filter("is_valid = True") \
    .select("valuation_id", "sub_vertical", "ev_bracket", "metric_type",
            "p25", "median_multiple", "p75", "deal_count",
            "source_year", "source_file", "ingestion_date")

print(f"Validated rows ready to merge: {validated_df.count()}")

Validated rows ready to merge: 404


In [0]:
validated_df.createOrReplaceTempView("staging_validated")

## Idempotent MERGE on Business Key

In [0]:
%sql
MERGE INTO silver_valuation_multiple AS target
USING staging_validated AS source
ON  target.sub_vertical = source.sub_vertical
AND target.metric_type = source.metric_type
AND (target.ev_bracket = source.ev_bracket OR (target.ev_bracket IS NULL AND source.ev_bracket IS NULL))
AND (target.source_year = source.source_year OR (target.source_year IS NULL AND source.source_year IS NULL))
WHEN MATCHED THEN
  UPDATE SET target.p25 = source.p25,
             target.median_multiple = source.median_multiple,
             target.p75 = source.p75,
             target.deal_count = source.deal_count,
             target.ingestion_date = source.ingestion_date
WHEN NOT MATCHED THEN
  INSERT (valuation_id, sub_vertical, ev_bracket, metric_type, p25, median_multiple, p75,
          deal_count, source_year, source_file, ingestion_date)
  VALUES (source.valuation_id, source.sub_vertical, source.ev_bracket, source.metric_type,
          source.p25, source.median_multiple, source.p75, source.deal_count,
          source.source_year, source.source_file, source.ingestion_date)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
404,404,0,0


## Verify Merge Result

In [0]:
display(spark.sql("SELECT COUNT(*) AS row_count FROM silver_valuation_multiple"))
display(spark.sql("DESCRIBE HISTORY silver_valuation_multiple").select("version", "timestamp", "operation", "operationMetrics"))

row_count
404


version,timestamp,operation,operationMetrics
5,2026-08-18T05:47:13Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 13394, numTargetBytesRemoved -> 13394, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 404, executionTimeMs -> 5851, materializeSourceTimeMs -> 519, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3609, numTargetRowsUpdated -> 404, numOutputRows -> 404, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 404, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1599)"
4,2026-08-17T14:57:02Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 13394, numTargetBytesRemoved -> 13394, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 404, executionTimeMs -> 4398, materializeSourceTimeMs -> 660, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1980, numTargetRowsUpdated -> 404, numOutputRows -> 404, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 404, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1717)"
3,2026-08-17T14:56:08Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 13394, numTargetBytesRemoved -> 13197, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 404, executionTimeMs -> 9973, materializeSourceTimeMs -> 1573, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 5084, numTargetRowsUpdated -> 404, numOutputRows -> 404, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 404, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3117)"
2,2026-08-17T06:52:15Z,WRITE,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 14687, numDeletionVectorsRemoved -> 0, numOutputRows -> 404, numOutputBytes -> 13197)"
1,2026-08-17T06:37:31Z,WRITE,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 14748, numDeletionVectorsRemoved -> 0, numOutputRows -> 481, numOutputBytes -> 14687)"
0,2026-08-17T05:33:59Z,WRITE,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 481, numOutputBytes -> 14748)"


In [0]:
history_df = spark.sql("DESCRIBE HISTORY silver_valuation_multiple")

display(history_df.filter("operation = 'MERGE'")
        .select("version", "timestamp",
                "operationMetrics.numTargetRowsUpdated",
                "operationMetrics.numTargetRowsInserted",
                "operationMetrics.numTargetRowsCopied",
                "operationMetrics.numOutputRows"))

version,timestamp,numTargetRowsUpdated,numTargetRowsInserted,numTargetRowsCopied,numOutputRows
5,2026-08-18T05:47:13Z,404,0,0,404
4,2026-08-17T14:57:02Z,404,0,0,404
3,2026-08-17T14:56:08Z,404,0,0,404


In [0]:
display(spark.sql("SELECT COUNT(*) AS row_count FROM silver_valuation_multiple"))

row_count
404
